<table style="width: 100%; border-collapse: collapse; border: none; background: #f8fafc; border-left: 6px solid #1e3a8a; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #0f172a; font-size: 2.1em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        Taller Hands-On: MercadoAndino — Clics Tardíos, Redes de Vendedores y Confianza en el Modelo 🌊🕸️
      </h1>
      <p style="margin: 6px 0 0 0; color: #1e3a8a; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Big Data
      </p>
      <p style="margin: 4px 0 0 0; color: #64748b; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #1e3a8a; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        Taller Práctico Evaluativo · Módulo 05
      </span><br>
      <span style="color: #64748b; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #2563eb; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

<div align="center" style="margin-top: 15px; margin-bottom: 15px;">
  <a href="https://colab.research.google.com/github/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/blob/main/Big%20Data/homeworks/05_Streaming_Grafos_Tendencias_Hands_On.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/>
  </a>
</div>

---
## Objetivos de Aprendizaje 🔎

Este taller consolida el **Módulo 05: Avances y Tendencias en Big Data** ([README del módulo](../05%20-%20Avances%20y%20Tendencias%20en%20Big%20Data/README.md)) con un problema **nuevo**: la tienda en línea **MercadoAndino** (ficticia) recibe **clics que llegan tarde y desordenados**, tiene una **red de vendedores que se recomiendan entre sí**, un **modelo en producción** que puede degradarse sin avisar y un **libro de entregas** que debe ser a prueba de alteraciones.

Al terminar serás capaz de:

1. **Contar un flujo por tiempo de evento** con ventanas *tumbling*, *sliding* y de sesión, y usar una **marca de agua** para decidir qué datos tardíos se descartan, verificando **conteos exactos** con casos de prueba.
2. **Calcular PageRank por iteración de potencias** (con nodos colgantes), verificarlo contra `networkx.pagerank` y **hallar componentes conexas** con el modelo *vertex-céntrico* BSP de Pregel.
3. **Vigilar la deriva de datos** con el **índice de estabilidad poblacional (PSI)** y **entrenar de forma federada** (FedAvg), comparándolo con el modelo centralizado.
4. **Construir una cadena de bloques mínima**: árbol de Merkle con pruebas de inclusión, bloques con prueba de trabajo y **detección de manipulación**.

> 📚 **Repasa si lo necesitas:** [Cuaderno 00: flujos y grafos](../05%20-%20Avances%20y%20Tendencias%20en%20Big%20Data/00_Stream_Processing_y_Graph_Processing.ipynb) · [Cuaderno 01: IA y *blockchain*](../05%20-%20Avances%20y%20Tendencias%20en%20Big%20Data/01_Tendencias_Emergentes_IA_y_Blockchain.ipynb).

---
## Instrucciones 📋

1. **Resuelve de forma autónoma:** en cada reto completa la celda de código marcada con `# TODO:`. Este cuaderno **no trae soluciones**.
2. **Autoverificación:** debajo de cada reto hay una celda con `assert`. Si tu solución es correcta imprime ✅; si no, el mensaje te dice qué revisar. No la modifiques.
3. **Puntaje:** cada reto indica su puntaje sugerido; el total del taller suma **100 puntos**.
4. Antes de entregar usa *Restart & Run All*: el cuaderno debe correr de principio a fin, sin errores y **sin `# TODO` pendientes**. Tarda menos de un minuto.
5. Todo funciona **sin internet y sin archivos externos** (datos sintéticos con semilla fija). Bibliotecas: `numpy`, `pandas`, `networkx` y `hashlib`. **No se usa *Spark***: las ventanas, la marca de agua, PageRank, Pregel, FedAvg y la cadena de bloques los implementas tú.
6. Los datos son **inventados**: los resultados describen el mecanismo, no el comportamiento de una tienda real.

In [ ]:
# ============================================================
# Preparación (no modificar): imports y datos sintéticos
# ============================================================
import copy
import hashlib
import math
import warnings
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import networkx as nx

warnings.filterwarnings("ignore")
SEMILLA = 55

# ---------- Reto 1: clics de usuarios. Cada evento es (usuario, t_evento, t_llegada), en segundos ----------
def generar_flujo(n=2400, seed=SEMILLA):
    rng = np.random.default_rng(seed)
    usuario = rng.integers(0, 30, n)
    t_ev = np.round(rng.uniform(0, 600, n), 2)
    demora = np.minimum(rng.exponential(2.5, n), 30)                       # la mayoría llega casi de inmediato...
    demora = np.where(rng.random(n) < 0.05, rng.uniform(20, 55, n), demora)  # ...y un 5 % llega muy tarde (< 60 s)
    t_ll = t_ev + demora
    return [(f"u{u:02d}", float(a), float(b)) for u, a, b in zip(usuario, t_ev, t_ll)]


EV = generar_flujo()

# Flujo pequeño, escrito a mano, con resultados que puedes verificar con lápiz (sirve para las pruebas)
EV_MANO = [("ana", 1, 2), ("ana", 4, 5), ("beto", 8, 9), ("ana", 12, 13), ("beto", 14, 15), ("beto", 22, 23),
           ("beto", 27, 28), ("ana", 31, 32), ("ana", 33, 34), ("ana", 17, 36), ("beto", 6, 41)]

# ---------- Reto 2: red de vendedores (quién recomienda a quién) y red de bodegas (quién está conectado con quién) ----------
N_NODOS = 60
rng_g = np.random.default_rng(SEMILLA + 1)
A_REC = rng_g.random((N_NODOS, N_NODOS)) < 0.045
np.fill_diagonal(A_REC, False)
A_REC[[7, 31], :] = False                                   # los vendedores 7 y 31 no recomiendan a nadie (nodos colgantes)
G = nx.DiGraph()
G.add_nodes_from(range(N_NODOS))
G.add_edges_from((int(i), int(j)) for i, j in zip(*np.nonzero(A_REC)))   # arista i -> j: «i recomienda a j»

# Red pequeña con respuesta conocida: 1,2,3 recomiendan a 0 y a 7; 4 y 5 solo a 0; 0 solo a 6; 6 y 7 devuelven a 0
G_MINI = nx.DiGraph([(1, 0), (2, 0), (3, 0), (4, 0), (5, 0), (1, 7), (2, 7), (3, 7), (0, 6), (6, 0), (7, 0)])


def generar_bodegas(seed=SEMILLA + 2):
    rng = np.random.default_rng(seed)
    aristas = [(i, i + 1) for i in range(11)]                                   # camino 0..11
    aristas += [(i, j) for i in range(12, 17) for j in range(i + 1, 17)]         # clique 12..16
    aristas += [(v, int(rng.integers(17, v))) for v in range(18, 41)]            # árbol aleatorio 17..40
    adj = {v: [] for v in range(44)}                                             # 41, 42 y 43 quedan aislados
    for a, b in aristas:
        adj[a].append(b)
        adj[b].append(a)
    return {v: sorted(vs) for v, vs in adj.items()}


adj_cc = generar_bodegas()

# ---------- Reto 3: deriva (PSI) y aprendizaje federado ----------
rng3 = np.random.default_rng(SEMILLA + 3)
X_REF = rng3.normal(0, 1, (5000, 2))                                            # datos de entrenamiento: columnas x1, x2
SEMANAS = [np.c_[rng3.normal(0.3 * w, 1, 2000), rng3.normal(0, 1, 2000)] for w in range(8)]   # x1 deriva 0,3 por semana


def sigmoide(z):
    return 1 / (1 + np.exp(-z))


def exactitud(w, X, y):
    return float(np.mean((sigmoide(X @ w) >= 0.5) == y))


rng4 = np.random.default_rng(SEMILLA + 4)
W_REAL = np.array([1.5, -2.0, 1.0, 0.5, -1.0, 0.3])                             # el último coeficiente es el sesgo
X_F = np.c_[rng4.normal(size=(2400, 5)), np.ones(2400)]
y_F = (rng4.random(2400) < sigmoide(X_F @ W_REAL)).astype(float)
X_tr, y_tr, X_te, y_te = X_F[:1400], y_F[:1400], X_F[1400:], y_F[1400:]
CORTES = np.cumsum([700, 400, 200, 100])[:-1]                                   # 4 clientes de tamaños distintos
clientes_iid = list(zip(np.split(X_tr, CORTES), np.split(y_tr, CORTES)))        # reparto al azar
_o = np.argsort(y_tr, kind="stable")                                            # reparto ordenado por etiqueta (no IID)
clientes_noiid = list(zip(np.split(X_tr[_o], CORTES), np.split(y_tr[_o], CORTES)))
LR, RONDAS = 0.5, 200

# ---------- Reto 4: libro de entregas ----------
def sha(texto):
    return hashlib.sha256(texto.encode("utf-8")).hexdigest()


rng5 = np.random.default_rng(SEMILLA + 5)
LOTES = [[f"pedido=P{100 * b + i}|monto={int(rng5.integers(20, 500)) * 1000}" for i in range(int(rng5.integers(3, 8)))] for b in range(6)]
DIF = 3                                                                          # dificultad: el hash debe empezar con «000»

print("Eventos del flujo:", len(EV), "| red de vendedores:", G.number_of_nodes(), "nodos y", G.number_of_edges(), "aristas")
print("Red de bodegas:", len(adj_cc), "nodos | Clientes federados:", [len(y) for _, y in clientes_iid], "| Lotes de entregas:", [len(l) for l in LOTES])

---
## Reto 1: Clics tardíos — ventanas, sesiones y marca de agua ⏱️

**Contexto:** `EV` es una lista de eventos `(usuario, t_evento, t_llegada)` en segundos: `t_evento` es cuándo ocurrió el clic y `t_llegada` cuándo lo recibió el sistema (siempre posterior). Un 5 % llega muy tarde. Vas a contar **por tiempo de evento** y a decidir cuánto esperar a los rezagados. `EV_MANO` es un flujo de 11 eventos para probar a mano.

**Enunciado** (todas las funciones reciben una lista de eventos como `EV`, en cualquier orden)
1. `conteo_tumbling(eventos, ancho)`: diccionario `{inicio: n}`. La ventana que contiene a `t` empieza en `(t // ancho) * ancho` y cubre $[inicio,\ inicio + ancho)$.
2. `conteo_sliding(eventos, ancho, paso)`: ventanas $[s,\ s + ancho)$ con $s$ múltiplo de `paso` y $s \ge 0$; un evento cuenta en **todas** las que lo contienen, o sea $t - ancho < s \le t$. Devuelve `{s: n}` solo con ventanas no vacías.
3. `sesiones(eventos, brecha)`: por usuario, ordena sus eventos por `t_evento`; una sesión nueva empieza cuando el salto con el evento anterior es **estrictamente mayor** que `brecha`. Devuelve `{usuario: [(inicio, fin, n), ...]}` con el primer y el último `t_evento` de cada sesión y su número de eventos.
4. `motor_marca_agua(eventos, ancho, retraso)`: procesa los eventos en **orden de `t_llegada`**. La **marca de agua** es `max(t_evento de los eventos ya recibidos) - retraso` (vale $-\infty$ al inicio). Un evento se **descarta** si el **final de su ventana *tumbling*** (`inicio + ancho`) es **menor o igual** que la marca vigente *antes* de recibirlo; si no, se cuenta. Devuelve `(conteos, descartados)`.
5. `completitud(retraso)`: fracción de los eventos de `EV` que el motor cuenta con `ancho = 30`. Calcula `tabla_completitud` (diccionario `{retraso: completitud}`) para `candidatos` y `retraso_min`, el **menor** candidato con completitud $\ge 0{,}99$.

**Criterios de aceptación:** conteos **exactos** en `EV_MANO` y coincidencia con un cálculo independiente en `EV`; con un retraso mayor o igual que la demora máxima no se descarta nada; la completitud nunca decrece al aumentar el retraso, y `retraso_min` no puede ser 0 (esperar cuesta latencia, no esperar cuesta datos).

**Puntaje sugerido:** 25 puntos.

In [ ]:
def conteo_tumbling(eventos, ancho):
    # TODO: {inicio_de_ventana: cantidad de eventos}, por TIEMPO DE EVENTO
    return None


def conteo_sliding(eventos, ancho, paso):
    # TODO: {s: n} para las ventanas [s, s + ancho) con s múltiplo de `paso` y s >= 0
    return None


def sesiones(eventos, brecha):
    # TODO: {usuario: [(inicio, fin, n), ...]}; nueva sesión si el salto es ESTRICTAMENTE mayor que `brecha`
    return None


def motor_marca_agua(eventos, ancho, retraso):
    # TODO: recorre los eventos por orden de t_llegada, lleva la marca de agua y descarta los demasiado tardíos
    # Devuelve (conteos, descartados): conteos = {inicio: n} de los eventos aceptados
    return None


def completitud(retraso):
    # TODO: fracción de EV contada por motor_marca_agua con ancho = 30
    return None


candidatos = [0, 5, 10, 20, 30, 45, 60, 90]
tabla_completitud = None   # {retraso: completitud} para cada candidato
retraso_min = None         # menor candidato con completitud >= 0.99

In [ ]:
# Autoverificación del Reto 1 (no modificar)
# --- Casos de prueba con respuesta exacta (EV_MANO) ---
assert conteo_tumbling(EV_MANO, 10) == {0: 4, 10: 3, 20: 2, 30: 2}, "conteo_tumbling(EV_MANO, 10) debe dar {0: 4, 10: 3, 20: 2, 30: 2}: cuenta por t_evento (no por llegada) con inicio = (t // ancho) * ancho."
assert conteo_sliding(EV_MANO, 10, 5) == {0: 4, 5: 4, 10: 3, 15: 2, 20: 2, 25: 3, 30: 2}, "conteo_sliding(EV_MANO, 10, 5) debe dar {0: 4, 5: 4, 10: 3, 15: 2, 20: 2, 25: 3, 30: 2}: un evento cuenta en todas las ventanas con t - ancho < s <= t y s >= 0."
assert sesiones(EV_MANO, 6) == {"ana": [(1, 4, 2), (12, 17, 2), (31, 33, 2)], "beto": [(6, 14, 3), (22, 27, 2)]}, "sesiones(EV_MANO, 6) no coincide: ordena por t_evento y recuerda que un salto IGUAL a la brecha NO abre sesión nueva (beto: 6, 8, 14)."
assert motor_marca_agua(EV_MANO, 10, 5) == ({0: 3, 10: 2, 20: 2, 30: 2}, 2), "motor_marca_agua(EV_MANO, 10, 5) debe dar ({0: 3, 10: 2, 20: 2, 30: 2}, 2): se descartan los eventos de t = 17 y t = 6 (llegan cuando la marca de agua ya pasó el final de su ventana)."
assert motor_marca_agua(EV_MANO, 10, 15) == ({0: 3, 10: 3, 20: 2, 30: 2}, 1), "motor_marca_agua(EV_MANO, 10, 15) debe dar ({0: 3, 10: 3, 20: 2, 30: 2}, 1): con más retraso el evento de t = 17 se salva."
assert motor_marca_agua(EV_MANO, 10, 25) == ({0: 4, 10: 3, 20: 2, 30: 2}, 0), "motor_marca_agua(EV_MANO, 10, 25) debe dar ({0: 4, 10: 3, 20: 2, 30: 2}, 0)."
assert motor_marca_agua([("a", 5, 1), ("a", 25, 2), ("a", 9, 3)], 10, 15) == ({0: 1, 20: 1}, 1), "Caso frontera: con retraso 15 la marca de agua vale 10 cuando llega el evento de t = 9 y su ventana termina EXACTAMENTE en 10: se descarta (final de ventana <= marca)."

# --- Comparación con un cálculo independiente sobre el flujo completo (EV) ---
_df = pd.DataFrame(EV, columns=["u", "t", "ll"])
_tum = (_df["t"] // 30 * 30).value_counts().to_dict()
assert conteo_tumbling(EV, 30) == _tum, "conteo_tumbling(EV, 30) no coincide con el conteo de pandas sobre el flujo completo."
_sl = {}
for _s in np.arange(0, 640, 5.0):
    _n = int(((_df["t"] >= _s) & (_df["t"] < _s + 20)).sum())
    if _n:
        _sl[_s] = _n
_mi = conteo_sliding(EV, 20, 5)
assert {float(k): v for k, v in _mi.items()} == _sl, "conteo_sliding(EV, 20, 5) no coincide con el conteo por fuerza bruta (ventanas [s, s + 20) con s múltiplo de 5)."
assert sum(_mi.values()) > len(EV), "Con ancho 20 y paso 5 cada evento debe contarse en varias ventanas: la suma de conteos supera el número de eventos."
_d = _df.sort_values(["u", "t"]).reset_index(drop=True)
_d["nueva"] = (_d.groupby("u")["t"].diff() > 8).astype(int)
_d["sid"] = _d.groupby("u")["nueva"].cumsum()
_ref = {u: [(g["t"].min(), g["t"].max(), len(g)) for _, g in sub.groupby("sid")] for u, sub in _d.groupby("u")}
_ses = sesiones(EV, 8)
assert _ses == _ref, "sesiones(EV, 8) no coincide con el cálculo por usuario y salto > 8 s sobre el flujo completo."
assert sum(n for v in _ses.values() for _, _, n in v) == len(EV), "Todos los eventos deben pertenecer a alguna sesión."

# --- Marca de agua: propiedades y comparación vectorizada ---
_a = _df.sort_values("ll").reset_index(drop=True)
_marca = np.concatenate([[-np.inf], np.maximum.accumulate(_a["t"].to_numpy())[:-1]]) - 20
_fin = (_a["t"] // 30 * 30 + 30).to_numpy()
_desc20 = int((_fin <= _marca).sum())
_c20, _d20 = motor_marca_agua(EV, 30, 20)
assert _d20 == _desc20, f"Con retraso 20 deberían descartarse {_desc20} eventos y tu motor descarta {_d20}."
assert sum(_c20.values()) + _d20 == len(EV), "Cada evento se cuenta o se descarta: conteos + descartados debe ser el total."
_demora_max = max(ll - t for _, t, ll in EV)
_c60, _d60 = motor_marca_agua(EV, 30, 60)
assert _demora_max < 60 and _d60 == 0 and _c60 == _tum, "Con un retraso mayor que la demora máxima de cualquier evento no debe descartarse nada y el resultado debe igualar al conteo tumbling por tiempo de evento."
assert motor_marca_agua(EV, 30, 0)[1] > 0, "Sin retraso (0 s) deben descartarse eventos tardíos."

# --- Completitud y retraso mínimo ---
assert isinstance(tabla_completitud, dict) and set(tabla_completitud) == set(candidatos), "tabla_completitud debe ser un diccionario con una clave por cada candidato."
_vals = [tabla_completitud[r] for r in candidatos]
assert all(abs(tabla_completitud[r] - (1 - motor_marca_agua(EV, 30, r)[1] / len(EV))) < 1e-12 for r in candidatos), "Cada completitud debe ser 1 - descartados / total, con ancho = 30."
assert all(b >= a - 1e-12 for a, b in zip(_vals, _vals[1:])), "La completitud no debe decrecer al aumentar el retraso."
assert tabla_completitud[90] == 1.0, "Con 90 s de retraso la completitud debe ser exactamente 1."
assert retraso_min in candidatos and tabla_completitud[retraso_min] >= 0.99 and all(tabla_completitud[r] < 0.99 for r in candidatos if r < retraso_min), "retraso_min debe ser el MENOR candidato con completitud >= 0.99."
assert retraso_min > 0, "Con retraso 0 se pierde más del 1 % de los eventos: retraso_min debe ser mayor que 0."
print(f"✅ Reto 1 superado: completitud con retraso 0 = {tabla_completitud[0]:.1%}; para llegar al 99 % hay que esperar {retraso_min} s (y {tabla_completitud[retraso_min]:.2%} de los clics quedan contados).")

---
## Reto 2: PageRank y componentes conexas con Pregel 🕸️

**Contexto:** en la red `G` (60 vendedores) una arista $i \to j$ significa «$i$ recomienda a $j$». Los vendedores 7 y 31 **no recomiendan a nadie** (nodos colgantes). Quieres medir la **importancia** de cada vendedor y, aparte, saber qué bodegas (`adj_cc`) quedan conectadas entre sí.

**Enunciado**
1. `pagerank_potencias(G, alfa=0.85, tol=1e-12, max_iter=1000)`: devuelve `(pr, iteraciones)` con `pr` un arreglo de NumPy en el orden de `sorted(G.nodes())`. Parte de $\mathbf{p}_0 = \tfrac1n\mathbf{1}$ y repite
$$\mathbf{p}_{t+1} = \alpha\,\bigl(\mathbf{p}_t P + \tfrac{1}{n}\,s_t\,\mathbf{1}\bigr) + \tfrac{1-\alpha}{n}\,\mathbf{1},\qquad s_t=\sum_{i\ \text{colgante}} p_t(i),$$
donde $P_{ij} = 1/\mathrm{grado\_salida}(i)$ si $i \to j$ (los colgantes reparten su puntaje **por igual entre todos**). Para cuando $\lVert \mathbf{p}_{t+1}-\mathbf{p}_t\rVert_1 <$ `tol`.
2. `pr, iters_pr = pagerank_potencias(G)` y `pr_mini, _ = pagerank_potencias(G_MINI)`. En `G_MINI` el nodo 6 recibe **una** recomendación (de la tienda estrella 0) y el nodo 7 recibe **tres**: ¿quién vale más?
3. `iteraciones`: diccionario `{alfa: iteraciones}` para $\alpha \in \{0{,}5;\ 0{,}85;\ 0{,}95\}$ sobre `G`.
4. `componentes_bsp(adj)`: modelo **vertex-céntrico** por superpasos. Cada vértice parte con su propio identificador como etiqueta; en el superpaso 0 todos **envían** su etiqueta a sus vecinos; después, cada vértice que **recibió mensajes** toma el mínimo entre su etiqueta y los mensajes, y **solo si bajó** reenvía la nueva etiqueta; termina cuando no quedan mensajes en tránsito. Devuelve `(etiquetas, n_super)` con `etiquetas = {vértice: etiqueta}` y `n_super` el número de superpasos ejecutados.

**Criterios de aceptación:** `pr` coincide con `networkx.pagerank` (tolerancia $10^{-7}$), suma 1 y es positivo; en `G_MINI` el nodo 6 supera al 7 **aunque tenga menos enlaces entrantes**; más $\alpha$ exige más iteraciones; las etiquetas de `componentes_bsp` agrupan **exactamente** las componentes de `networkx`, cada una etiquetada con su menor identificador, y `n_super` está entre el diámetro máximo y ese diámetro + 3.

**Puntaje sugerido:** 20 puntos.

In [ ]:
def pagerank_potencias(G, alfa=0.85, tol=1e-12, max_iter=1000):
    nodos = sorted(G.nodes())
    # TODO: matriz de adyacencia A = nx.to_numpy_array(G, nodelist=nodos); P = reparto por grado de salida;
    #       trata los colgantes (reparto uniforme) e itera hasta que la diferencia L1 sea menor que tol
    # Devuelve (pr, iteraciones)
    return None


def componentes_bsp(adj):
    # TODO: superpasos con mensajes {vértice: [etiquetas recibidas]}; devuelve (etiquetas, n_super)
    return None


pr = iters_pr = None        # PageRank de G y número de iteraciones
pr_mini = None              # PageRank de G_MINI
iteraciones = None          # {alfa: iteraciones} para alfa en 0.5, 0.85 y 0.95
etiquetas = n_super = None  # resultado de componentes_bsp(adj_cc)

In [ ]:
# Autoverificación del Reto 2 (no modificar)
assert pr is not None and len(pr) == N_NODOS, "pr debe ser el arreglo de PageRank de G, con un valor por nodo (en el orden de sorted(G.nodes()))."
assert abs(float(np.sum(pr)) - 1) < 1e-9 and np.all(np.asarray(pr) > 0), "El PageRank debe sumar 1 y ser positivo en todos los nodos: revisa el reparto de los nodos colgantes y el término (1 - alfa) / n."
_nx = nx.pagerank(G, alpha=0.85, tol=1e-12, max_iter=1000)
_ref = np.array([_nx[i] for i in range(N_NODOS)])
assert np.allclose(pr, _ref, atol=1e-7), "pr no coincide con networkx.pagerank: los nodos colgantes (7 y 31) reparten su puntaje por igual entre TODOS los nodos."
assert iters_pr is not None and 5 < iters_pr < 1000, "iters_pr debe ser el número de iteraciones hasta converger (más de 5 y menos que max_iter)."
_nxm = nx.pagerank(G_MINI, alpha=0.85, tol=1e-12, max_iter=1000)
assert pr_mini is not None and np.allclose(pr_mini, [_nxm[i] for i in range(8)], atol=1e-7), "pr_mini debe coincidir con networkx.pagerank sobre G_MINI."
assert G_MINI.in_degree(6) < G_MINI.in_degree(7) and pr_mini[6] > pr_mini[7], "En G_MINI el nodo 6 tiene MENOS enlaces entrantes que el 7 pero debe valer MÁS: la importancia se hereda de quien te recomienda."
assert isinstance(iteraciones, dict) and set(iteraciones) == {0.5, 0.85, 0.95}, "iteraciones debe tener una clave por cada alfa: 0.5, 0.85 y 0.95."
assert iteraciones[0.5] < iteraciones[0.85] < iteraciones[0.95], "Con un alfa mayor la convergencia es más lenta: iteraciones debe crecer con alfa."
assert iteraciones[0.85] == iters_pr, "iteraciones[0.85] debe coincidir con iters_pr."

_Gcc = nx.Graph(adj_cc)
_comps = {frozenset(c) for c in nx.connected_components(_Gcc)}
assert etiquetas is not None and set(etiquetas) == set(adj_cc), "etiquetas debe tener una entrada por cada vértice de adj_cc."
_grupos = defaultdict(set)
for _v, _e in etiquetas.items():
    _grupos[_e].add(_v)
assert {frozenset(g) for g in _grupos.values()} == _comps, "Los vértices con la misma etiqueta deben formar EXACTAMENTE una componente conexa de networkx."
assert all(etiquetas[v] == min(c) for c in _comps for v in c), "Cada componente debe quedar etiquetada con su menor identificador de vértice (propaga el mínimo)."
assert sorted(len(c) for c in _comps) == [1, 1, 1, 5, 12, 24], "La red de bodegas tiene seis componentes: 24, 12, 5 y tres aisladas."
_diam = max(nx.diameter(_Gcc.subgraph(c)) for c in _comps)
assert n_super is not None and _diam <= n_super <= _diam + 3, f"n_super = {n_super} no es razonable: la etiqueta mínima recorre el diámetro ({_diam}) en tantos superpasos; para ahí cuando no quedan mensajes."
print(f"✅ Reto 2 superado: PageRank de G converge en {iters_pr} iteraciones (alfa 0.5/0.85/0.95 → {iteraciones[0.5]}/{iteraciones[0.85]}/{iteraciones[0.95]}) y Pregel halló {len(_comps)} componentes en {n_super} superpasos (diámetro {_diam}).")

---
## Reto 3: Un modelo en producción — deriva (PSI) y entrenamiento federado 📉🤝

**Contexto:** MercadoAndino despliega un modelo que usa dos variables, `x1` y `x2`. Cada semana llegan 2 000 registros nuevos y quieres saber **cuándo hay que reentrenar**. Además, cuatro tiendas aliadas (de 700, 400, 200 y 100 registros) quieren entrenar **sin compartir sus datos**.

**Enunciado — A. Deriva con PSI**
1. `psi(esperado, actual, n_cajones=10)`: define los cajones con los **deciles de `esperado`** (bordes internos `np.quantile(esperado, linspace(0, 1, n_cajones + 1)[1:-1])`); calcula la proporción $e_i$ de `esperado` y $a_i$ de `actual` en cada cajón, **sustituye por $10^{-4}$** las proporciones menores que $10^{-4}$ y devuelve
$$\mathrm{PSI}=\sum_i (a_i - e_i)\,\ln\frac{a_i}{e_i}.$$
2. `psi_x1` y `psi_x2`: listas con el PSI de cada una de las 8 semanas de `SEMANAS` (columnas 0 y 1) frente a `X_REF`.
3. `semana_alerta`: la **primera** semana con `psi_x1 > 0.25` (la regla práctica: > 0,25 = cambio importante).

**Enunciado — B. FedAvg** (regresión logística con sesgo ya incluido como última columna de `X`)
4. `grad_logistico(w, X, y)`: gradiente medio de la pérdida logística, $\tfrac1n X^{\top}(\sigma(Xw)-y)$ (usa `sigmoide`).
5. `fedavg(clientes, rondas, lr, epocas, ponderar=True)`: parte de $w=\mathbf 0$; en cada ronda cada cliente copia $w$ y da `epocas` pasos de descenso de gradiente **con sus datos**; el servidor promedia los $w_k$ con pesos $n_k/n$ (o pesos iguales si `ponderar=False`).
6. `w_central`: descenso **centralizado** (`RONDAS` pasos con `LR` sobre `X_tr`, `y_tr`). `w_fed1 = fedavg(clientes_iid, RONDAS, LR, 1)` y `w_simple` igual pero con `ponderar=False`. Luego `w_fed5 = fedavg(clientes_iid, RONDAS // 5, LR, 5)` y `w_noiid = fedavg(clientes_noiid, RONDAS // 5, LR, 5)`, y sus exactitudes en prueba: `acc_central`, `acc_fed5`, `acc_noiid` (usa `exactitud`).

**Criterios de aceptación:** `psi` coincide con la definición, es 0 con datos idénticos y crece con el corrimiento; `psi_x2` nunca supera 0,1 y la alerta salta entre las semanas 2 y 3; con una época por ronda y pesos $n_k/n$, FedAvg **es idéntico** al centralizado (promediar gradientes pesados = gradiente completo) y el promedio sin pesos **no**; con 5 épocas locales, tanto IID como no IID quedan a menos de 0,10 de exactitud del centralizado.

**Puntaje sugerido:** 25 puntos.

In [ ]:
def psi(esperado, actual, n_cajones=10):
    # TODO: cajones por deciles de `esperado`, proporciones (mínimo 1e-4) y suma de (a - e) * ln(a / e)
    return None


psi_x1 = None        # lista con el PSI semanal de x1 (columna 0 de cada semana)
psi_x2 = None        # lista con el PSI semanal de x2 (columna 1)
semana_alerta = None # primera semana con psi_x1 > 0.25


def grad_logistico(w, X, y):
    # TODO: gradiente medio de la pérdida logística
    return None


def fedavg(clientes, rondas, lr, epocas, ponderar=True):
    # TODO: w = ceros; en cada ronda, cada cliente da `epocas` pasos de gradiente desde w; promedia (pesos n_k / n o iguales)
    return None


w_central = None   # descenso centralizado: RONDAS pasos con LR sobre X_tr, y_tr
w_fed1 = None      # FedAvg IID, 1 época por ronda, RONDAS rondas
w_simple = None    # igual que w_fed1 pero ponderar=False
w_fed5 = None      # FedAvg IID, 5 épocas por ronda, RONDAS // 5 rondas
w_noiid = None     # FedAvg no IID, 5 épocas por ronda, RONDAS // 5 rondas
acc_central = acc_fed5 = acc_noiid = None   # exactitudes en prueba (X_te, y_te)

In [ ]:
# Autoverificación del Reto 3 (no modificar)
def _psi_ref(e, a, k=10):
    b = np.quantile(e, np.linspace(0, 1, k + 1)[1:-1])
    pe = np.clip(np.histogram(e, bins=np.r_[-np.inf, b, np.inf])[0] / len(e), 1e-4, None)
    pa = np.clip(np.histogram(a, bins=np.r_[-np.inf, b, np.inf])[0] / len(a), 1e-4, None)
    return float(np.sum((pa - pe) * np.log(pa / pe)))

_x = X_REF[:, 0]
_v0 = psi(_x, _x)
assert _v0 is not None and abs(_v0) < 1e-12, "psi(x, x) debe ser 0: con datos idénticos todas las proporciones coinciden."
assert abs(psi(_x, _x + 8) - _psi_ref(_x, _x + 8)) < 1e-9, "psi no coincide con la definición cuando hay cajones vacíos: sustituye por 1e-4 las proporciones menores que 1e-4."
assert all(abs(psi(_x, S[:, 0]) - _psi_ref(_x, S[:, 0])) < 1e-9 for S in SEMANAS), "psi no coincide con la definición (deciles de `esperado`, sustitución por 1e-4 y suma de (a - e) * ln(a / e))."
assert psi(_x, _x + 0.2) < psi(_x, _x + 0.5) < psi(_x, _x + 1.0), "El PSI debe crecer cuando el corrimiento aumenta."
assert isinstance(psi_x1, list) and len(psi_x1) == 8 and isinstance(psi_x2, list) and len(psi_x2) == 8, "psi_x1 y psi_x2 deben ser listas con 8 valores (una por semana)."
assert np.allclose(psi_x1, [_psi_ref(X_REF[:, 0], S[:, 0]) for S in SEMANAS]) and np.allclose(psi_x2, [_psi_ref(X_REF[:, 1], S[:, 1]) for S in SEMANAS]), "psi_x1 / psi_x2 deben comparar cada semana (columna 0 / 1) contra X_REF."
assert max(psi_x2) < 0.1 and psi_x1[0] < 0.1, "x2 no deriva (y x1 en la semana 0 tampoco): su PSI debe quedar por debajo de 0,1."
assert psi_x1[7] > psi_x1[1], "x1 deriva cada semana: su PSI de la semana 7 debe superar al de la semana 1."
assert semana_alerta == next(w for w, v in enumerate(psi_x1) if v > 0.25) and 2 <= semana_alerta <= 3, "semana_alerta debe ser la primera semana con psi_x1 > 0.25 (con esta deriva, la 2 o la 3)."

_g = grad_logistico(np.full(6, 0.1), X_tr, y_tr)
assert _g is not None and np.allclose(_g, X_tr.T @ (sigmoide(X_tr @ np.full(6, 0.1)) - y_tr) / len(y_tr), atol=1e-10), "grad_logistico debe ser X.T @ (sigmoide(X @ w) - y) / n."
assert w_central is not None and np.shape(w_central) == (6,), "w_central debe ser un vector de 6 coeficientes (5 variables + sesgo)."
_w = np.zeros(6)
for _ in range(RONDAS):
    _w = _w - LR * (X_tr.T @ (sigmoide(X_tr @ _w) - y_tr) / len(y_tr))
assert np.allclose(w_central, _w, atol=1e-9), "w_central debe ser RONDAS pasos de descenso de gradiente con LR desde ceros sobre X_tr, y_tr."
assert w_fed1 is not None and np.allclose(w_fed1, w_central, atol=1e-9), "Con 1 época local y pesos n_k / n, FedAvg debe ser IDÉNTICO al centralizado (la suma ponderada de gradientes es el gradiente completo)."
assert w_simple is not None and np.abs(w_simple - w_central).max() > 1e-3, "Con ponderar=False (pesos iguales) y clientes de tamaños distintos, el resultado NO debe coincidir con el centralizado."
assert acc_central is not None and abs(acc_central - exactitud(w_central, X_te, y_te)) < 1e-12 and acc_central > 0.7, "acc_central es la exactitud de w_central en (X_te, y_te) y debe superar 0,7."
assert acc_fed5 is not None and abs(acc_fed5 - exactitud(w_fed5, X_te, y_te)) < 1e-12 and abs(acc_fed5 - acc_central) < 0.10, "acc_fed5 (IID, 5 épocas) debe ser la exactitud de w_fed5 y quedar a menos de 0,10 del centralizado."
assert acc_noiid is not None and abs(acc_noiid - exactitud(w_noiid, X_te, y_te)) < 1e-12 and acc_noiid > acc_central - 0.10, "acc_noiid debe ser la exactitud de w_noiid y no quedar más de 0,10 por debajo del centralizado."
print(f"✅ Reto 3 superado: la alerta de deriva salta en la semana {semana_alerta} (PSI = {psi_x1[semana_alerta]:.2f}) | exactitud: centralizado {acc_central:.3f}, FedAvg IID {acc_fed5:.3f}, FedAvg no IID {acc_noiid:.3f}")

---
## Reto 4: El libro de entregas a prueba de alteraciones ⛓️

**Contexto:** MercadoAndino registra sus entregas en bloques. Quieres que cualquier alteración posterior **se note** y que probar que un pedido está en un bloque no exija mostrar todos los demás. Usarás `sha(texto)` (SHA-256 en hexadecimal) y `LOTES` (6 listas de transacciones de texto).

**Enunciado**
1. `raiz_merkle(datos)`: las hojas son `sha(d)` de cada dato; en cada nivel, si el número de nodos es **impar** duplica el último; el padre de $(a, b)$ es `sha(a + b)` (concatenación de textos). La raíz es el único nodo final (con un solo dato, `sha(dato)`).
2. `prueba_merkle(datos, i)`: lista de pares `(hash_hermano, lado)` desde la hoja hasta la raíz, donde `lado` es `"izq"` o `"der"` según la posición **del hermano**. `verificar_prueba(dato, prueba, raiz)`: recalcula la raíz desde el dato y devuelve `True`/`False`.
3. `crear_bloque(indice, transacciones, hash_prev, dificultad)`: devuelve un diccionario con las llaves `indice`, `transacciones`, `raiz` (la raíz de Merkle), `hash_prev`, `nonce` y `hash`, donde `hash = sha(f"{indice}|{raiz}|{hash_prev}|{nonce}")` y el `nonce` es el **menor entero** $\ge 0$ con el que el `hash` **empieza con `dificultad` ceros**.
4. `construir_cadena(lotes, dificultad)`: un bloque por lote; el primero usa `hash_prev = "0" * 64`. Guarda `cadena = construir_cadena(LOTES, DIF)`.
5. `primer_bloque_invalido(cadena, dificultad)`: índice del primer bloque que **no cumple** alguna regla (raíz que no corresponde a sus transacciones, `hash` que no corresponde a su cabecera, `hash` sin los ceros exigidos, o `hash_prev` distinto del `hash` del bloque anterior) o `None` si todo es válido.
6. Ataques sobre copias de `cadena` que cambian la **primera transacción del bloque 3** por `"pedido=P9999|monto=1"`: **(a)** sin tocar nada más; **(b)** recalculando también la `raiz`; **(c)** reconstruyendo el bloque 3 con `crear_bloque` (hash válido), sin tocar los siguientes; **(d)** reconstruyendo los bloques 3, 4 y 5 encadenados. Guarda `cadena_a`, ..., `cadena_d` y los resultados `res_a`, ..., `res_d` de `primer_bloque_invalido`.

**Criterios de aceptación:** Merkle y bloques coinciden con cálculos independientes; todas las pruebas de inclusión válidas (y falsas con datos alterados); `cadena` es válida; los ataques se detectan en los bloques **3, 3, 4** y solo el ataque (d) —el que rehace **todos** los bloques posteriores— pasa la verificación.

**Puntaje sugerido:** 15 puntos.

In [ ]:
def raiz_merkle(datos):
    # TODO: hojas = sha(dato); duplica el último si el nivel es impar; padre = sha(izq + der)
    return None


def prueba_merkle(datos, i):
    # TODO: [(hash_hermano, "izq" | "der"), ...] desde la hoja i hasta la raíz
    return None


def verificar_prueba(dato, prueba, raiz):
    # TODO: recalcula la raíz desde sha(dato) usando la prueba; devuelve True si coincide con `raiz`
    return None


def crear_bloque(indice, transacciones, hash_prev, dificultad):
    # TODO: raiz = raiz_merkle(...); busca el menor nonce >= 0 cuyo hash empiece con `dificultad` ceros
    return None


def construir_cadena(lotes, dificultad):
    # TODO: un bloque por lote, encadenados; el primero con hash_prev = "0" * 64
    return None


def primer_bloque_invalido(cadena, dificultad):
    # TODO: índice del primer bloque inválido (raíz, hash, ceros o enlace con el anterior) o None
    return None


cadena = None   # construir_cadena(LOTES, DIF)

NUEVA_TX = "pedido=P9999|monto=1"
cadena_a = cadena_b = cadena_c = cadena_d = None   # copias manipuladas (copy.deepcopy(cadena))
res_a = res_b = res_c = res_d = None                # primer_bloque_invalido de cada copia

In [ ]:
# Autoverificación del Reto 4 (no modificar)
_a, _b, _c = "pedido=1", "pedido=2", "pedido=3"
assert raiz_merkle([_a]) == sha(_a), "Con un solo dato la raíz es sha(dato)."
assert raiz_merkle([_a, _b]) == sha(sha(_a) + sha(_b)), "Con dos datos la raíz es sha(sha(a) + sha(b))."
assert raiz_merkle([_a, _b, _c]) == sha(sha(sha(_a) + sha(_b)) + sha(sha(_c) + sha(_c))), "Con tres datos, el tercero se empareja con una copia de sí mismo: sha(sha(sha(a)+sha(b)) + sha(sha(c)+sha(c)))."
assert raiz_merkle(["x", "y", "z"]) != raiz_merkle(["x", "z", "y"]), "El orden de los datos debe cambiar la raíz."
for _n in (1, 2, 3, 5, 8, 11):
    _datos = [f"tx{k}" for k in range(_n)]
    _r = raiz_merkle(_datos)
    for _i in range(_n):
        _p = prueba_merkle(_datos, _i)
        assert _p is not None and verificar_prueba(_datos[_i], _p, _r) is True, f"La prueba de inclusión del dato {_i} (de {_n}) debe verificarse con la raíz."
        assert verificar_prueba(_datos[_i] + "!", _p, _r) is False, "Un dato alterado NO debe verificarse con la prueba original."
    assert len(prueba_merkle(_datos, 0)) == math.ceil(math.log2(_n)) if _n > 1 else prueba_merkle(_datos, 0) == [], "La prueba debe tener ceil(log2 n) pasos: es logarítmica, no hay que mostrar los demás datos."
assert verificar_prueba("tx0", prueba_merkle([f"tx{k}" for k in range(5)], 1), raiz_merkle([f"tx{k}" for k in range(5)])) is False, "La prueba del dato 1 no sirve para el dato 0."

_b0 = crear_bloque(7, ["p1", "p2", "p3"], "ab" * 32, 2)
assert _b0 is not None and set(_b0) == {"indice", "transacciones", "raiz", "hash_prev", "nonce", "hash"}, "El bloque debe tener las llaves indice, transacciones, raiz, hash_prev, nonce y hash."
assert _b0["raiz"] == raiz_merkle(["p1", "p2", "p3"]) and _b0["hash"] == sha(f"7|{_b0['raiz']}|{'ab' * 32}|{_b0['nonce']}"), "hash = sha(f\"{indice}|{raiz}|{hash_prev}|{nonce}\")."
assert _b0["hash"].startswith("00") and all(not sha(f"7|{_b0['raiz']}|{'ab' * 32}|{k}").startswith("00") for k in range(_b0["nonce"])), "El nonce debe ser el MENOR entero >= 0 con el que el hash empieza con los ceros exigidos."

assert cadena is not None and len(cadena) == len(LOTES) == 6, "cadena debe tener un bloque por lote (6)."
assert cadena[0]["hash_prev"] == "0" * 64 and all(cadena[i]["hash_prev"] == cadena[i - 1]["hash"] for i in range(1, 6)), "Cada bloque debe guardar el hash del anterior (el primero, 64 ceros)."
assert all(b["transacciones"] == l and b["hash"].startswith("0" * DIF) for b, l in zip(cadena, LOTES)), "Cada bloque debe contener las transacciones de su lote y un hash que cumpla la prueba de trabajo."
assert primer_bloque_invalido(cadena, DIF) is None, "La cadena original es válida: primer_bloque_invalido debe devolver None."

for _nombre, _ch in [("a", cadena_a), ("b", cadena_b), ("c", cadena_c), ("d", cadena_d)]:
    assert _ch is not None and _ch is not cadena and _ch[3]["transacciones"][0] == NUEVA_TX, f"cadena_{_nombre} debe ser una copia (copy.deepcopy) con la primera transacción del bloque 3 cambiada."
    assert cadena[3]["transacciones"][0] != NUEVA_TX, "No modifiques la cadena original: trabaja sobre copias."
assert cadena_a[3]["raiz"] == cadena[3]["raiz"] and cadena_a[3]["hash"] == cadena[3]["hash"], "En el ataque (a) solo cambia el texto de la transacción."
assert cadena_b[3]["raiz"] == raiz_merkle(cadena_b[3]["transacciones"]) and cadena_b[3]["hash"] == cadena[3]["hash"], "En el ataque (b) se recalcula la raíz pero NO el hash."
assert cadena_c[3]["hash"].startswith("0" * DIF) and cadena_c[3]["hash"] != cadena[3]["hash"] and cadena_c[4] == cadena[4], "En el ataque (c) se rehace SOLO el bloque 3 (con crear_bloque) y el bloque 4 queda intacto."
assert all(cadena_d[i]["hash"] != cadena[i]["hash"] for i in (3, 4, 5)) and all(cadena_d[i] == cadena[i] for i in (0, 1, 2)), "En el ataque (d) se rehacen los bloques 3, 4 y 5 y los anteriores quedan intactos."
assert (res_a, res_b, res_c) == (3, 3, 4), f"Los ataques (a), (b) y (c) deben detectarse en los bloques 3, 3 y 4 (obtuviste {res_a}, {res_b}, {res_c}): revisa qué regla viola cada uno."
assert res_d is None, "El ataque (d) rehace TODOS los bloques posteriores con su prueba de trabajo, así que la cadena resultante es válida: manipular sale caro, pero no imposible."
print(f"✅ Reto 4 superado: ataques detectados en los bloques {res_a}, {res_b} y {res_c}; el ataque (d) obligó a minar de nuevo {sum(cadena_d[i]['nonce'] + 1 for i in (3, 4, 5))} hashes (3 bloques) y la raíz de Merkle de un lote de {len(LOTES[0])} pedidos se prueba con {len(prueba_merkle(LOTES[0], 0))} pasos.")

---
## Reto 5: Conclusiones 📝

**Puntaje sugerido:** 15 puntos (sin código). Escribe **5 a 8 líneas** en la celda de abajo interpretando *tus* resultados. Debe responder:

- ¿Qué ganas y qué pierdes al subir el `retraso` de la marca de agua? Usa tu `retraso_min`, la completitud con retraso 0 y la diferencia entre contar por tiempo de evento y por tiempo de llegada.
- En la red de vendedores, ¿por qué el nodo 6 de `G_MINI` vale más que el 7 con menos recomendaciones? ¿Qué te dice que ese resultado sea el de `networkx`?
- ¿Por qué el PSI de `x1` alertó en la semana `semana_alerta` y el de `x2` nunca? ¿Qué **no** puede detectar el PSI?
- ¿Por qué FedAvg con 1 época es idéntico al centralizado y con 5 épocas solo muy parecido (compara también los coeficientes, no solo la exactitud)? ¿Qué cambió con datos no IID?
- ¿Qué detectó el verificador de la cadena en los ataques (a), (b) y (c), y qué **no** garantiza una cadena de bloques aunque (d) sea costoso?

---
#### ✍️ Tus conclusiones

_Escribe aquí tus 5-8 líneas. Reemplaza este texto por tu análisis._

---
### 🧠 Rúbrica de evaluación

| Criterio | Puntos |
|---|---|
| Reto 1: ventanas *tumbling*, *sliding* y de sesión, marca de agua y retraso mínimo con casos de prueba exactos | 25 |
| Reto 2: PageRank por iteración de potencias (verificado con `networkx`) y componentes conexas con Pregel/BSP | 20 |
| Reto 3: PSI y alerta de deriva, FedAvg frente al modelo centralizado (IID y no IID) | 25 |
| Reto 4: árbol de Merkle, bloques con prueba de trabajo y detección de manipulación | 15 |
| Reto 5: conclusiones argumentadas con tus cifras (5-8 líneas) | 15 |
| **Total** | **100** |

### ✅ Checklist de entrega

- [ ] Reinicié el kernel y ejecuté *Run All* sin errores.
- [ ] No quedan `# TODO` ni variables en `None`.
- [ ] Las cuatro celdas de autoverificación imprimen ✅.
- [ ] Conté los eventos por **tiempo de evento** y justifiqué el retraso elegido.
- [ ] Verifiqué mi PageRank contra `networkx` y mis componentes contra `nx.connected_components`.
- [ ] Comparé FedAvg con el centralizado y expliqué por qué el promedio debe ser ponderado.
- [ ] Escribí mis conclusiones con cifras de mis propios resultados.

---
<div align="center">
  <p style="font-size: 0.9em; color: #64748b;">
    © 2026 <b>Universidad Santo Tomás — Seccional Tunja</b><br>
    <i>Especialización en Ciencia de Datos | Asignatura: Big Data</i>
  </p>
</div>